In [1]:
!pip install optuna mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 814.0/814.0 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 14.5 MB/s eta 0:00:00


In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import optuna
import mlflow
import mlflow.keras

In [3]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# convertir imágenes a vectores
x_train = x_train.reshape(-1, 784) / 255.0
x_test = x_test.reshape(-1, 784) / 255.0

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [4]:
def create_model(trial):

    model = keras.Sequential()

    n_layers = trial.suggest_int("n_layers", 1, 3)

    for i in range(n_layers):
        units = trial.suggest_int(f"units_{i}", 32, 256)
        activation = trial.suggest_categorical("activation", ["relu", "tanh"])

        model.add(layers.Dense(units, activation=activation))

    model.add(layers.Dense(10, activation="softmax"))

    optimizer_name = trial.suggest_categorical("optimizer", ["adam", "sgd"])

    model.compile(
        optimizer=optimizer_name,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [5]:
def objective(trial):

    model = create_model(trial)

    history = model.fit(
        x_train, y_train,
        validation_split=0.2,
        epochs=5,
        batch_size=128,
        verbose=0
    )

    loss, acc = model.evaluate(x_test, y_test, verbose=0)

    return acc

In [6]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("Best parameters:", study.best_params)

[I 2026-03-06 16:53:25,983] A new study created in memory with name: no-name-534de9d2-d10f-4b2b-aaa6-5b97bca10f26
[I 2026-03-06 16:53:38,214] Trial 0 finished with value: 0.9713000059127808 and parameters: {'n_layers': 2, 'units_0': 104, 'activation': 'relu', 'units_1': 69, 'optimizer': 'adam'}. Best is trial 0 with value: 0.9713000059127808.
[I 2026-03-06 16:53:49,258] Trial 1 finished with value: 0.8996000289916992 and parameters: {'n_layers': 1, 'units_0': 147, 'activation': 'tanh', 'optimizer': 'sgd'}. Best is trial 0 with value: 0.9713000059127808.
[I 2026-03-06 16:53:59,877] Trial 2 finished with value: 0.909600019454956 and parameters: {'n_layers': 2, 'units_0': 113, 'activation': 'relu', 'units_1': 141, 'optimizer': 'sgd'}. Best is trial 0 with value: 0.9713000059127808.
[I 2026-03-06 16:54:10,346] Trial 3 finished with value: 0.9685999751091003 and parameters: {'n_layers': 2, 'units_0': 53, 'activation': 'tanh', 'units_1': 201, 'optimizer': 'adam'}. Best is trial 0 with value:

Best parameters: {'n_layers': 3, 'units_0': 252, 'activation': 'relu', 'units_1': 243, 'units_2': 122, 'optimizer': 'adam'}


In [7]:
model = keras.Sequential([
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    x_train,
    y_train,
    epochs=5,
    validation_split=0.2
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8680 - loss: 0.4671 - val_accuracy: 0.9529 - val_loss: 0.1626
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9594 - loss: 0.1359 - val_accuracy: 0.9665 - val_loss: 0.1134
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9740 - loss: 0.0843 - val_accuracy: 0.9647 - val_loss: 0.1162
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9813 - loss: 0.0623 - val_accuracy: 0.9754 - val_loss: 0.0836
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9875 - loss: 0.0440 - val_accuracy: 0.9754 - val_loss: 0.0854


In [9]:
model = keras.Sequential([
    layers.Dense(128, activation="relu",
                 kernel_regularizer=regularizers.l1(0.001)),
    layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(x_train, y_train, epochs=5, validation_split=0.2)

NameError: name 'regularizers' is not defined

In [10]:
model = keras.Sequential([
    layers.Dense(128, activation="relu",
                 kernel_regularizer=regularizers.l1(0.001)),
    layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(x_train, y_train, epochs=5, validation_split=0.2)

NameError: name 'regularizers' is not defined

In [11]:
model = keras.Sequential([
    layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l1(0.001)
    ),
    layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(x_train, y_train, epochs=5, validation_split=0.2)

NameError: name 'regularizers' is not defined

In [12]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import regularizers

# cargar dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train.reshape(-1,784)/255.0
x_test = x_test.reshape(-1,784)/255.0


# modelo con regularización L1
model = keras.Sequential([
    layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l1(0.001)
    ),
    layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    x_train,
    y_train,
    epochs=5,
    validation_split=0.2
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8412 - loss: 1.6287 - val_accuracy: 0.9208 - val_loss: 0.6356
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9129 - loss: 0.6390 - val_accuracy: 0.9318 - val_loss: 0.5283
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.9269 - loss: 0.5376 - val_accuracy: 0.9362 - val_loss: 0.4862
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9334 - loss: 0.4908 - val_accuracy: 0.9443 - val_loss: 0.4417
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.9410 - loss: 0.4459 - val_accuracy: 0.9474 - val_loss: 0.4228


In [13]:
model = keras.Sequential([
    layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l2(0.001)
    ),
    layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    x_train,
    y_train,
    epochs=5,
    validation_split=0.2
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8645 - loss: 0.6105 - val_accuracy: 0.9527 - val_loss: 0.2634
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9519 - loss: 0.2521 - val_accuracy: 0.9600 - val_loss: 0.2184
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9622 - loss: 0.2145 - val_accuracy: 0.9631 - val_loss: 0.2110
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9675 - loss: 0.1887 - val_accuracy: 0.9657 - val_loss: 0.1997
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.9695 - loss: 0.1801 - val_accuracy: 0.9703 - val_loss: 0.1789


In [14]:
model = keras.Sequential([
    layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l1_l2(0.001)
    ),
    layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    x_train,
    y_train,
    epochs=5,
    validation_split=0.2
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.8340 - loss: 1.6540 - val_accuracy: 0.9160 - val_loss: 0.6381
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9100 - loss: 0.6391 - val_accuracy: 0.9323 - val_loss: 0.5299
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9261 - loss: 0.5385 - val_accuracy: 0.9430 - val_loss: 0.4743
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9339 - loss: 0.4897 - val_accuracy: 0.9478 - val_loss: 0.4420
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9388 - loss: 0.4501 - val_accuracy: 0.9452 - val_loss: 0.4239


In [15]:
model = keras.Sequential([
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    x_train,
    y_train,
    epochs=5,
    validation_split=0.2
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8298 - loss: 0.5648 - val_accuracy: 0.9551 - val_loss: 0.1539
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.9456 - loss: 0.1829 - val_accuracy: 0.9666 - val_loss: 0.1175
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9581 - loss: 0.1406 - val_accuracy: 0.9697 - val_loss: 0.1014
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9655 - loss: 0.1132 - val_accuracy: 0.9710 - val_loss: 0.0927
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9701 - loss: 0.0972 - val_accuracy: 0.9726 - val_loss: 0.0916


In [16]:
model = keras.Sequential([
    layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l1_l2(0.001)
    ),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    x_train,
    y_train,
    epochs=5,
    validation_split=0.2
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8119 - loss: 1.7855 - val_accuracy: 0.9104 - val_loss: 0.7290
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.8898 - loss: 0.7732 - val_accuracy: 0.9232 - val_loss: 0.6417
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9025 - loss: 0.7000 - val_accuracy: 0.9317 - val_loss: 0.5972
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.9087 - loss: 0.6633 - val_accuracy: 0.9390 - val_loss: 0.5768
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9126 - loss: 0.6478 - val_accuracy: 0.9415 - val_loss: 0.5401
